# 위고비 치료 확산과 정기사망보험금

SELECT와 유사한 가상 가입자 1,000명에서 치료 시작이 늘어날 때 기대 사망보험금이 어떻게 달라지는지 계산하는 분석    

⚠️ *보험료 수입·사업비·준비금은 포함하지 않으므로 결과는 회사 이익이 아니라 **기대 사망보험금 지급액과 현재가치의 차이**다.*

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

## 1. 근거값

기초 위험은 SELECT 위약군의 전체 사망 사건수와 관찰 인년으로 잡는다.   
`사망 사건수 / 인년`은 사망확률이 아니라 발생률이다. 이를 일정한 사력으로 간주해 기간별 사망확률로 바꾼다(약간 지수사망법칙 느낌)

$$\mu_0 = D_0 / PY_0, \qquad {}_tq_0 = 1-\exp(-\mu_0t).$$

이 가정에서는 나이가 들어도 기초 사망력이 변하지 않는다. 

In [2]:
source_data = pd.DataFrame([
    {"arm": "semaglutide", "deaths": 375, "person_years": 29_283},
    {"arm": "placebo", "deaths": 458, "person_years": 29_112},
]).set_index("arm")

source_data["death_rate_per_year"] = source_data["deaths"] / source_data["person_years"]
source_data["annual_q_constant_hazard"] = -np.expm1(-source_data["death_rate_per_year"])

evidence = {
    "mortality_hr": 0.81,
    "mortality_hr_ci_lower": 0.71,
    "mortality_hr_ci_upper": 0.93,
    "mean_follow_up_months": 39.8,
    "deaths_and_person_years_source": (
        "https://www.ncbi.nlm.nih.gov/books/NBK619926/"
        "table/tr8269868382084182_ch01_t19/?report=objectonly"
    ),
    "hr_source": "https://doi.org/10.1056/NEJMoa2307563",
}

mu0 = float(source_data.loc["placebo", "death_rate_per_year"])
display(source_data)

,deaths,person_years,death_rate_per_year,annual_q_constant_hazard
arm,,,,
semaglutide,375,29283,0.0128,0.0127
placebo,458,29112,0.0157,0.0156


## 2. 임시 계약 조건과 치료 시작 경로

- 잔여 보장기간 10년, 사망보험금 1억 원인 동일한 정기보험 계약을 가정한다. 
- 최초에는 모두 생존·유지하고 치료 이력이 없다. 해지는 이번 계산에서 제외
- 아래 **연 시작확률 1%와 3%는 계산을 위한 임시값**
- 시작확률의 분모는 그 시점까지 살아 있는 미시작자
- 매달 초 신규 시작자가 생긴다고 두고, 연 확률을 일정 시작력 가정으로 월 확률로 바꾼다

$$p_{\mathrm{start,month}}=1-(1-p_{\mathrm{start,year}})^{1/12}.$$

In [3]:
settings = {
    "initial_policies": 1_000,
    "coverage_years": 10,
    "death_benefit_krw": 100_000_000,
    "annual_discount_rate": 0.03,
    "effect_years": 3,
}

months = np.arange(settings["coverage_years"] * 12)

# 나중에는 월별로 다른 시작확률을 넣을 수 있도록 경로를 따로 둔다.
start_paths = pd.DataFrame({
    "month_index": months,
    "annual_start_low": np.full(len(months), 0.01),
    "annual_start_high": np.full(len(months), 0.03),
})
start_paths["policy_year"] = start_paths["month_index"] // 12 + 1

hr_cases = {"no_effect": 1.0, "select_exploratory": evidence["mortality_hr"]}

display(pd.Series(settings, name="value").to_frame())
display(start_paths.head(12))

,value
initial_policies,"1,000.0000"
coverage_years,10.0000
death_benefit_krw,"100,000,000.0000"
annual_discount_rate,0.0300
effect_years,3.0000


,month_index,annual_start_low,annual_start_high,policy_year
0,0,0.0100,0.0300,1
1,1,0.0100,0.0300,1
2,2,0.0100,0.0300,1
3,3,0.0100,0.0300,1
4,4,0.0100,0.0300,1
5,5,0.0100,0.0300,1
6,6,0.0100,0.0300,1
7,7,0.0100,0.0300,1
8,8,0.0100,0.0300,1
9,9,0.0100,0.0300,1


## 3. 월별 코호트 계산

매달 다음 순서로 계산한다.

1. 살아 있는 미시작자에서 신규 시작자를 분리한다.
2. 미시작자와 시작 시점별 코호트의 월 사망확률을 계산한다.
3. 사망자를 제외한 생존자를 다음 달로 넘긴다.
4. 예상 사망자 수에 보험금을 곱하고 현재가치로 할인한다.

- 치료군의 사망확률은 적용 중인 HR로 사망력을 조정해서 구한다. 
- 효과 기간이 월 중간에 끝나는 경우에는 적용·미적용 기간을 나눠 누적 사망력을 계산한다. 
- 효과가 끝나도 생존자는 만기까지 추적한다.

In [4]:
def project_mortality(annual_start_path, mortality_hr, settings, baseline_hazard):
    annual_start_path = np.asarray(annual_start_path, dtype=float)
    n_months = len(annual_start_path)
    monthly_start_path = 1 - (1 - annual_start_path) ** (1 / 12)

    untreated_alive = float(settings["initial_policies"])
    # 배열의 위치는 최초 치료 시작월, 값은 현재 살아 있는 해당 코호트의 인원이다.
    treated_alive = np.zeros(n_months)
    effect_months = settings["effect_years"] * 12
    monthly_q0 = -np.expm1(-baseline_hazard / 12)
    records = []

    for t in range(n_months):
        alive_open = untreated_alive + treated_alive.sum()
        untreated_open = untreated_alive

        new_starters = untreated_open * monthly_start_path[t]
        untreated_alive -= new_starters
        treated_alive[t] += new_starters

        elapsed_months = t - np.arange(n_months)
        active_fraction = np.clip(effect_months - elapsed_months, 0, 1)
        active_fraction[elapsed_months < 0] = 0

        # 1개월 중 효과 적용 비중을 이용해 월 누적 사망력을 계산한다.
        hazard_multiplier = 1 + (mortality_hr - 1) * active_fraction
        treated_monthly_q = -np.expm1(-baseline_hazard * hazard_multiplier / 12)

        untreated_deaths = untreated_alive * monthly_q0
        treated_deaths_by_cohort = treated_alive * treated_monthly_q
        treated_deaths = treated_deaths_by_cohort.sum()
        total_deaths = untreated_deaths + treated_deaths

        # 생존자 수가 이후 보험금에도 반영되도록 두 집단을 각각 갱신한다.
        untreated_alive -= untreated_deaths
        treated_alive -= treated_deaths_by_cohort
        alive_close = untreated_alive + treated_alive.sum()

        payment_time_years = (t + 0.5) / 12
        discount_factor = (1 + settings["annual_discount_rate"]) ** (-payment_time_years)
        death_claim = total_deaths * settings["death_benefit_krw"]

        records.append({
            "month_index": t,
            "policy_month": t + 1,
            "policy_year": t // 12 + 1,
            "annual_start_probability": annual_start_path[t],
            "monthly_start_probability": monthly_start_path[t],
            "alive_open": alive_open,
            "untreated_open": untreated_open,
            "new_starters": new_starters,
            "deaths_untreated": untreated_deaths,
            "deaths_started": treated_deaths,
            "deaths_total": total_deaths,
            "alive_close": alive_close,
            "untreated_close": untreated_alive,
            "started_alive_close": treated_alive.sum(),
            "death_claim_krw": death_claim,
            "death_claim_pv_krw": death_claim * discount_factor,
        })

    return pd.DataFrame(records)

## 4. 기준 시나리오 실행

낮은 확산과 높은 확산을 각각 무효과·SELECT 탐색적 효과 아래 계산  
+) `monthly_results`는 다음 시각화와 상세 확인에 사용할 월별 결과

In [5]:
runs = []
for hr_case, hr in hr_cases.items():
    for uptake_case, column in [("low", "annual_start_low"), ("high", "annual_start_high")]:
        result = project_mortality(
            annual_start_path=start_paths[column],
            mortality_hr=hr,
            settings=settings,
            baseline_hazard=mu0,
        )
        result.insert(0, "uptake_case", uptake_case)
        result.insert(0, "hr_case", hr_case)
        result["mortality_hr"] = hr
        runs.append(result)

monthly_results = pd.concat(runs, ignore_index=True)
display(monthly_results.head(12))

,hr_case,uptake_case,month_index,policy_month,policy_year,annual_start_probability,monthly_start_probability,alive_open,untreated_open,new_starters,deaths_untreated,deaths_started,deaths_total,alive_close,untreated_close,started_alive_close,death_claim_krw,death_claim_pv_krw,mortality_hr
0,no_effect,low,0,1,1,0.0100,0.0008,"1,000.0000","1,000.0000",0.8372,1.3091,0.0011,1.3102,998.6898,997.8537,0.8361,"131,016,964.8234","130,855,701.4614",1.0000
1,no_effect,low,1,2,1,0.0100,0.0008,998.6898,997.8537,0.8354,1.3063,0.0022,1.3085,997.3814,995.7121,1.6693,"130,845,310.3727","130,362,748.5868",1.0000
2,no_effect,low,2,3,1,0.0100,0.0008,997.3814,995.7121,0.8336,1.3035,0.0033,1.3067,996.0746,993.5751,2.4996,"130,673,880.8185","129,871,652.7389",1.0000
3,no_effect,low,3,4,1,0.0100,0.0008,996.0746,993.5751,0.8318,1.3007,0.0044,1.3050,994.7696,991.4426,3.3270,"130,502,675.8660","129,382,406.9221",1.0000
4,no_effect,low,4,5,1,0.0100,0.0008,994.7696,991.4426,0.8300,1.2979,0.0054,1.3033,993.4663,989.3147,4.1516,"130,331,695.2211","128,895,004.1669",1.0000
5,no_effect,low,5,6,1,0.0100,0.0008,993.4663,989.3147,0.8282,1.2951,0.0065,1.3016,992.1647,987.1914,4.9733,"130,160,938.5898","128,409,437.5304",1.0000
6,no_effect,low,6,7,1,0.0100,0.0008,992.1647,987.1914,0.8265,1.2923,0.0076,1.2999,990.8648,985.0726,5.7921,"129,990,405.6787","127,925,700.0956",1.0000
7,no_effect,low,7,8,1,0.0100,0.0008,990.8648,985.0726,0.8247,1.2895,0.0087,1.2982,989.5666,982.9584,6.6082,"129,820,096.1946","127,443,784.9715",1.0000
8,no_effect,low,8,9,1,0.0100,0.0008,989.5666,982.9584,0.8229,1.2868,0.0097,1.2965,988.2701,980.8488,7.4213,"129,650,009.8448","126,963,685.2934",1.0000
9,no_effect,low,9,10,1,0.0100,0.0008,988.2701,980.8488,0.8211,1.2840,0.0108,1.2948,986.9753,978.7436,8.2317,"129,480,146.3370","126,485,394.2221",1.0000


## 5. 연도별 결과와 계약기간 전체 결과

연간 사망자·보험금은 월별 값을 합산하고, 생존자는 연초·연말 값을 사용. 이미 월별로 할인한 현재가치를 다시 할인하지 않는다.

- `annual_results`는 연도별 추이, `summary`는 계약기간 전체 결과, `comparison`은 같은 HR 가정에서 확산 경로만 바꾼 차이
- 비교표의 감소액은 **낮은 확산 − 높은 확산**으로 정의 (음수도 그대로 남김)

In [6]:
annual_results = (
    monthly_results
    .groupby(["hr_case", "uptake_case", "policy_year"], as_index=False, sort=False)
    .agg(
        alive_open=("alive_open", "first"),
        new_starters=("new_starters", "sum"),
        deaths_total=("deaths_total", "sum"),
        alive_close=("alive_close", "last"),
        death_claim_krw=("death_claim_krw", "sum"),
        death_claim_pv_krw=("death_claim_pv_krw", "sum"),
    )
)

summary = (
    monthly_results
    .groupby(["hr_case", "uptake_case"], as_index=False, sort=False)
    .agg(
        total_starters=("new_starters", "sum"),
        total_deaths=("deaths_total", "sum"),
        survivors_at_maturity=("alive_close", "last"),
        total_claim_krw=("death_claim_krw", "sum"),
        total_claim_pv_krw=("death_claim_pv_krw", "sum"),
    )
)
summary["total_claim_억원"] = summary["total_claim_krw"] / 100_000_000
summary["total_claim_pv_억원"] = summary["total_claim_pv_krw"] / 100_000_000

low = summary.loc[summary["uptake_case"].eq("low")].set_index("hr_case")
high = summary.loc[summary["uptake_case"].eq("high")].set_index("hr_case")
comparison = pd.DataFrame({
    "additional_starters": high["total_starters"] - low["total_starters"],
    "deaths_difference_low_minus_high": low["total_deaths"] - high["total_deaths"],
    "claim_difference_krw": low["total_claim_krw"] - high["total_claim_krw"],
    "claim_pv_difference_krw": low["total_claim_pv_krw"] - high["total_claim_pv_krw"],
})
comparison["claim_pv_difference_억원"] = comparison["claim_pv_difference_krw"] / 100_000_000

display(annual_results.loc[
    annual_results["hr_case"].eq("select_exploratory")
].reset_index(drop=True))
display(summary)
display(comparison)

,hr_case,uptake_case,policy_year,alive_open,new_starters,deaths_total,alive_close,death_claim_krw,death_claim_pv_krw
0,select_exploratory,low,1,"1,000.0000",9.9284,15.5933,984.4067,"1,559,325,761.8303","1,536,571,099.2869"
1,select_exploratory,low,2,984.4067,9.6757,15.3212,969.0855,"1,532,120,949.6241","1,465,789,446.6505"
2,select_exploratory,low,3,969.0855,9.4294,15.0545,954.0310,"1,505,450,496.8869","1,398,323,808.2445"
3,select_exploratory,low,4,954.0310,9.1894,14.8084,939.2226,"1,480,840,822.0697","1,335,396,856.2943"
4,select_exploratory,low,5,939.2226,8.9555,14.5794,924.6432,"1,457,938,021.5284","1,276,450,008.0843"
5,select_exploratory,low,6,924.6432,8.7275,14.3539,910.2894,"1,435,387,320.2917","1,220,103,381.3534"
6,select_exploratory,low,7,910.2894,8.5054,14.1318,896.1575,"1,413,183,359.7018","1,166,242,374.1775"
7,select_exploratory,low,8,896.1575,8.2889,13.9132,882.2443,"1,391,320,861.2496","1,114,757,430.3337"
8,select_exploratory,low,9,882.2443,8.0779,13.6979,868.5464,"1,369,794,625.4125","1,065,543,817.4330"
9,select_exploratory,low,10,868.5464,7.8723,13.4860,855.0604,"1,348,599,530.5085","1,018,501,414.7916"


,hr_case,uptake_case,total_starters,total_deaths,survivors_at_maturity,total_claim_krw,total_claim_pv_krw,total_claim_억원,total_claim_pv_억원
0,no_effect,low,88.6505,145.5723,854.4277,"14,557,234,239.3182","12,651,742,345.3948",145.5723,126.5174
1,no_effect,high,244.0920,145.5723,854.4277,"14,557,234,239.3183","12,651,742,345.3948",145.5723,126.5174
2,select_exploratory,low,88.6505,144.9396,855.0604,"14,493,961,749.1036","12,597,679,636.6498",144.9396,125.9768
3,select_exploratory,high,244.0920,143.8117,856.1883,"14,381,168,974.1205","12,500,717,772.2819",143.8117,125.0072


,additional_starters,deaths_difference_low_minus_high,claim_difference_krw,claim_pv_difference_krw,claim_pv_difference_억원
hr_case,,,,,
no_effect,155.4416,-0.0000,-0.0000,-0.0000,-0.0000
select_exploratory,155.4416,1.1279,"112,792,774.9831","96,961,864.3678",0.9696
